In [66]:
import logging
import sys
from datetime import datetime

from pyspark.sql import SparkSession
import os
os.environ['AWS_REGION'] = 'eu-north-1'
# Logging configuration
formatter = logging.Formatter('[%(asctime)s] %(levelname)s @ line %(lineno)d: %(message)s')
handler = logging.StreamHandler(sys.stdout)
handler.setLevel(logging.INFO)
handler.setFormatter(formatter)
logger = logging.getLogger()
logger.setLevel(logging.INFO)
logger.addHandler(handler)


# Application-specific variables
dt_string = datetime.now().strftime("%Y_%m_%d_%H_%M_%S")
AppName = "EmployeeDataS3TableJob"



In [67]:
input_csv_path = "s3://input-files-csv-spark/"
s3table_arn = "arn:aws:s3tables:eu-north-1:491085426235:bucket/tablesemployees"
namespace = "doeks_namespace"
table_name = "employee_s3_table"
full_table_name = f"s3tablesbucket.{namespace}.{table_name}"

In [76]:
spark = (SparkSession
    .builder
    .appName(f"{AppName}_{dt_string}")
    .config("spark.sql.extensions", "org.apache.iceberg.spark.extensions.IcebergSparkSessionExtensions")
    .config("spark.sql.catalog.s3tablesbucket", "org.apache.iceberg.spark.SparkCatalog")
    .config("spark.sql.catalog.s3tablesbucket.catalog-impl", "software.amazon.s3tables.iceberg.S3TablesCatalog")
    .config("spark.sql.catalog.s3tablesbucket.warehouse", s3table_arn)
    .config('spark.hadoop.fs.s3.impl', "org.apache.hadoop.fs.s3a.S3AFileSystem")
    .config("spark.sql.defaultCatalog", "s3tablesbucket")
    .config("spark.hadoop.fs.s3a.connection.timeout", "1200000")
    .config("spark.hadoop.fs.s3a.path.style.access", "true")
    .config("spark.hadoop.fs.s3a.connection.maximum", "200")
    .config("spark.hadoop.fs.s3a.fast.upload", "true")
    .config("spark.hadoop.fs.s3a.readahead.range", "256K")
    .config("spark.hadoop.fs.s3a.input.fadvise", "random")
    .config("spark.hadoop.fs.s3a.aws.credentials.provider", "org.apache.hadoop.fs.s3a.EnvironmentVariableCredentialsProvider")
    .config("spark.hadoop.fs.s3a.aws.credentials.provider", "org.apache.hadoop.fs.s3a.SimpleAWSCredentialsProvider")
    .config("spark.hadoop.fs.s3a.endpoint", "eu-north-1.amazonaws.com")
    .config("com.amazonaws.services.s3.enableV4", "true")
    .config("spark.hadoop.fs.s3a.impl", "org.apache.hadoop.fs.s3a.S3AFileSystem") 
    .config("spark.hadoop.fs.s3a.aws.credentials.provider", "com.amazonaws.auth.DefaultAWSCredentialsProviderChain") 
    .getOrCreate())

spark.sparkContext.setLogLevel("DEBUG")
logger.info("Spark session initialized successfully")

[2025-01-18 19:45:35,673] INFO @ line 25: Spark session initialized successfully
[2025-01-18 19:45:35,673] INFO @ line 25: Spark session initialized successfully
[2025-01-18 19:45:35,673] INFO @ line 25: Spark session initialized successfully
[2025-01-18 19:45:35,673] INFO @ line 25: Spark session initialized successfully
[2025-01-18 19:45:35,673] INFO @ line 25: Spark session initialized successfully
[2025-01-18 19:45:35,673] INFO @ line 25: Spark session initialized successfully
[2025-01-18 19:45:35,673] INFO @ line 25: Spark session initialized successfully


In [39]:
namespace = "doeks_namespace"
table_name = "employee_s3_table"
full_table_name = f"s3tablesbucket.{namespace}.{table_name}"

# Step 1: Create namespace if not exists
logger.info(f"Creating namespace: {namespace}")
spark.sql(f"CREATE NAMESPACE IF NOT EXISTS s3tablesbucket.{namespace}")

[2025-01-18 18:50:07,000] INFO @ line 6: Creating namespace: doeks_namespace
[2025-01-18 18:50:07,000] INFO @ line 6: Creating namespace: doeks_namespace
[2025-01-18 18:50:07,000] INFO @ line 6: Creating namespace: doeks_namespace
[2025-01-18 18:50:07,000] INFO @ line 6: Creating namespace: doeks_namespace
[2025-01-18 18:50:07,000] INFO @ line 6: Creating namespace: doeks_namespace


DataFrame[]

In [69]:
result = spark.sql(f"SHOW NAMESPACES LIKE '{namespace}'").collect()
print(result)


[Row(namespace='doeks_namespace')]


In [70]:
# Step 2: Read input CSV data
logger.info(f"Reading employee data from input CSV: {input_csv_path}")
employee_df = spark.read.csv('employee_data.csv', header=True, inferSchema=True)

[2025-01-18 19:23:40,250] INFO @ line 2: Reading employee data from input CSV: s3://input-files-csv-spark/
[2025-01-18 19:23:40,250] INFO @ line 2: Reading employee data from input CSV: s3://input-files-csv-spark/
[2025-01-18 19:23:40,250] INFO @ line 2: Reading employee data from input CSV: s3://input-files-csv-spark/
[2025-01-18 19:23:40,250] INFO @ line 2: Reading employee data from input CSV: s3://input-files-csv-spark/
[2025-01-18 19:23:40,250] INFO @ line 2: Reading employee data from input CSV: s3://input-files-csv-spark/
[2025-01-18 19:23:40,250] INFO @ line 2: Reading employee data from input CSV: s3://input-files-csv-spark/
[2025-01-18 19:23:40,250] INFO @ line 2: Reading employee data from input CSV: s3://input-files-csv-spark/


In [71]:
logger.info("Previewing employee data schema")
employee_df.printSchema()

logger.info("Previewing first 10 records from the input data")
employee_df.show(10, truncate=False)

[2025-01-18 19:23:46,648] INFO @ line 1: Previewing employee data schema
[2025-01-18 19:23:46,648] INFO @ line 1: Previewing employee data schema
[2025-01-18 19:23:46,648] INFO @ line 1: Previewing employee data schema
[2025-01-18 19:23:46,648] INFO @ line 1: Previewing employee data schema
[2025-01-18 19:23:46,648] INFO @ line 1: Previewing employee data schema
[2025-01-18 19:23:46,648] INFO @ line 1: Previewing employee data schema
[2025-01-18 19:23:46,648] INFO @ line 1: Previewing employee data schema
root
 |-- id: integer (nullable = true)
 |-- name: string (nullable = true)
 |-- level: string (nullable = true)
 |-- salary: double (nullable = true)

[2025-01-18 19:23:46,685] INFO @ line 4: Previewing first 10 records from the input data
[2025-01-18 19:23:46,685] INFO @ line 4: Previewing first 10 records from the input data
[2025-01-18 19:23:46,685] INFO @ line 4: Previewing first 10 records from the input data
[2025-01-18 19:23:46,685] INFO @ line 4: Previewing first 10 records f

In [46]:
logger.info(f"Creating/Replacing and writing data to table: {full_table_name}")
(employee_df.writeTo(full_table_name)
            .using("iceberg")
            .createOrReplace())

[2025-01-18 18:53:59,738] INFO @ line 1: Creating/Replacing and writing data to table: s3tablesbucket.doeks_namespace.employee_s3_table
[2025-01-18 18:53:59,738] INFO @ line 1: Creating/Replacing and writing data to table: s3tablesbucket.doeks_namespace.employee_s3_table
[2025-01-18 18:53:59,738] INFO @ line 1: Creating/Replacing and writing data to table: s3tablesbucket.doeks_namespace.employee_s3_table
[2025-01-18 18:53:59,738] INFO @ line 1: Creating/Replacing and writing data to table: s3tablesbucket.doeks_namespace.employee_s3_table
[2025-01-18 18:53:59,738] INFO @ line 1: Creating/Replacing and writing data to table: s3tablesbucket.doeks_namespace.employee_s3_table


In [62]:
logger.info(f"Reading data back from Iceberg table: {full_table_name}")
iceberg_data_df = spark.read.format("iceberg").load(full_table_name)

logger.info("Previewing first 10 records from the Iceberg table")
iceberg_data_df.show(10, truncate=False)

[2025-01-18 19:15:01,414] INFO @ line 1: Reading data back from Iceberg table: s3tablesbucket.doeks_namespace.employee_s3_table
[2025-01-18 19:15:01,414] INFO @ line 1: Reading data back from Iceberg table: s3tablesbucket.doeks_namespace.employee_s3_table
[2025-01-18 19:15:01,414] INFO @ line 1: Reading data back from Iceberg table: s3tablesbucket.doeks_namespace.employee_s3_table
[2025-01-18 19:15:01,414] INFO @ line 1: Reading data back from Iceberg table: s3tablesbucket.doeks_namespace.employee_s3_table
[2025-01-18 19:15:01,414] INFO @ line 1: Reading data back from Iceberg table: s3tablesbucket.doeks_namespace.employee_s3_table
[2025-01-18 19:15:01,414] INFO @ line 1: Reading data back from Iceberg table: s3tablesbucket.doeks_namespace.employee_s3_table
[2025-01-18 19:15:01,461] INFO @ line 4: Previewing first 10 records from the Iceberg table
[2025-01-18 19:15:01,461] INFO @ line 4: Previewing first 10 records from the Iceberg table
[2025-01-18 19:15:01,461] INFO @ line 4: Preview

In [75]:
spark.sql(f"SELECT COUNT(*) FROM {full_table_name}").show()


+--------+
|count(1)|
+--------+
|   10000|
+--------+



In [ ]:
spark.stop()

In [85]:
access_key = os.environ.get('AWS_ACCESS_KEY_ID')
secret_key = os.environ.get('AWS_SECRET_ACCESS_KEY')
region = os.environ.get('AWS_DEFAULT_REGION')

spark_2 = (SparkSession
    .builder
    .appName(f"{AppName}_{dt_string}")
    .config("spark.hadoop.fs.s3.awsAccessKeyId", access_key)
    .config("spark.hadoop.fs.s3n.awsAccessKeyId", access_key)
    .config("spark.hadoop.fs.s3a.access.key", access_key)
    .config("spark.hadoop.fs.s3.awsSecretAccessKey", secret_key)
    .config("spark.hadoop.fs.s3n.awsSecretAccessKey", secret_key)
    .config("spark.hadoop.fs.s3a.secret.key", secret_key)
    .config("spark.hadoop.fs.s3n.impl", "org.apache.hadoop.fs.s3native.NativeS3FileSystem")
    .config("spark.hadoop.fs.s3a.impl", "org.apache.hadoop.fs.s3a.S3AFileSystem")
    .config("spark.hadoop.fs.s3.impl", "org.apache.hadoop.fs.s3.S3FileSystem")
    .config("spark.hadoop.fs.s3a.endpoint", f"s3.{region}.amazonaws.com")
    .getOrCreate())


In [87]:
input_csv_path2 = "s3a://input-files-csv-spark/employee_data.csv"

employee_df_2 = spark.read.csv(input_csv_path2, header=True, inferSchema=True)

Py4JJavaError: An error occurred while calling o351.csv.
: org.apache.hadoop.fs.s3a.impl.InstantiationIOException: `s3a://input-files-csv-spark/employee_data.csv': Class org.apache.hadoop.fs.s3a.EnvironmentVariableCredentialsProvider instantiation exception java.lang.ClassNotFoundException: org.apache.hadoop.fs.s3a.EnvironmentVariableCredentialsProvider (configuration key fs.s3a.aws.credentials.provider): org.apache.hadoop.fs.s3a.EnvironmentVariableCredentialsProvider
	at org.apache.hadoop.fs.s3a.impl.InstantiationIOException.instantiationException(InstantiationIOException.java:178)
	at org.apache.hadoop.fs.s3a.S3AUtils.getInstanceFromReflection(S3AUtils.java:695)
	at org.apache.hadoop.fs.s3a.auth.CredentialProviderListFactory.createAWSV2CredentialProvider(CredentialProviderListFactory.java:307)
	at org.apache.hadoop.fs.s3a.auth.CredentialProviderListFactory.buildAWSProviderList(CredentialProviderListFactory.java:253)
	at org.apache.hadoop.fs.s3a.auth.CredentialProviderListFactory.crea